# AEZ-26 Phase 2G — Driver–Response Statistical Synthesis and Robustness

This notebook uses:

- Phase 2D event-driver characterization;
- Phase 2F FINAL response metrics.

It **does not change** the 39 flash-drought events.

Main outputs:
- event-level statistical analysis table;
- primary driver-response associations;
- secondary atmospheric associations;
- event-physics associations;
- compound-driver-class comparisons;
- cross-response coupling;
- robustness sensitivities;
- cohort/QC summaries;
- figures;
- one ZIP audit package.

### Statistical design
- Spearman rank correlation
- 4,999 permutation p-values
- 2,000 bootstrap confidence intervals
- Benjamini–Hochberg FDR within pre-specified families
- permutation Kruskal–Wallis for four driver classes


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
from scipy.stats import rankdata
import shutil

BASE_DIR = Path('/content/drive/MyDrive/AEZ26_Final_Research_Data')
PHASE2_DIR = BASE_DIR / 'Phase2'

PHASE2D_DIR = PHASE2_DIR / 'Phase2D'
PHASE2F_DIR = PHASE2_DIR / 'Phase2F_FINAL'
PHASE2G_DIR = PHASE2_DIR / 'Phase2G'
FIG_DIR = PHASE2G_DIR / 'Figures'

PHASE2G_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

DRIVER_FILE = PHASE2D_DIR / 'AEZ26_Phase2D_Event_Driver_Characterization.csv'
RESPONSE_FILE = PHASE2F_DIR / 'AEZ26_Phase2F_Cross_Response_Event_Synthesis.csv'

for p in [DRIVER_FILE, RESPONSE_FILE]:
    print(p.name, '->', p.exists())
    if not p.exists():
        raise FileNotFoundError(f'Missing required input: {p}')

N_PERM = 4999
N_BOOT = 2000

print('Permutation iterations:', N_PERM)
print('Bootstrap iterations:', N_BOOT)
print('Phase 2G output folder:', PHASE2G_DIR)


In [ ]:
drivers = pd.read_csv(
    DRIVER_FILE,
    parse_dates=['onset_mid_date']
)

responses = pd.read_csv(
    RESPONSE_FILE,
    parse_dates=['onset_start_date', 'event_end_date']
)

assert len(drivers) == 39
assert len(responses) == 39
assert drivers['event_id'].is_unique
assert responses['event_id'].is_unique
assert set(drivers['event_id']) == set(responses['event_id'])

analysis = drivers.merge(
    responses,
    on='event_id',
    how='inner',
    validate='one_to_one',
    suffixes=('_driver', '')
)

assert len(analysis) == 39
assert analysis['event_id'].is_unique

print('Merged event analysis table:', analysis.shape)
print('Input gate: PASS')


## 1. Derived predictor and response-direction variables

Response depth is sign-oriented so that **larger positive values mean a stronger decline**.

No response value is clipped at zero.


In [ ]:
# Primary forcing strengths
analysis['P_deficit_strength_z'] = (
    -analysis['P_standardized_anomaly_z']
)

analysis['PE_excess_strength_z'] = (
    analysis['PE_standardized_anomaly_z']
)

analysis['VPD_excess_strength_z'] = (
    analysis['VPD_standardized_anomaly_z']
)

analysis['temperature_excess_strength_z'] = (
    analysis['temperature_standardized_anomaly_z']
)

# Secondary atmospheric strengths
analysis['radiation_excess_strength_z'] = (
    analysis['net_radiation_standardized_anomaly_z']
)

analysis['dry_air_RH_strength_z'] = (
    -analysis['RH_standardized_anomaly_z']
)

analysis['wind_excess_strength_z'] = (
    analysis['wind_standardized_anomaly_z']
)

# Primary response depths
analysis['MOD16_ETPET_response_depth_z'] = (
    -analysis['minimum_ETPET_drop_from_prebaseline_z']
)

analysis['MOD13_NDVI_response_depth_z'] = (
    -analysis['minimum_NDVI_drop_from_prebaseline_z']
)

analysis['MOD13_EVI_response_depth_z'] = (
    -analysis['minimum_EVI_drop_from_prebaseline_z']
)

analysis['GRACE_TWSA_response_depth_z'] = (
    -analysis['minimum_TWSA_drop_from_prebaseline_z']
)

analysis['JRC_water_response_depth_z'] = (
    -analysis['minimum_water_fraction_drop_from_prebaseline_z']
)

# QA sensitivity response depths
analysis['MOD16_ETPET_response_depth_strictQA_z'] = -(
    analysis['minimum_ETPET_z_strictQA_onset_to_post64']
    - analysis['prebaseline_ETPET_z_strictQA']
)

analysis['MOD13_NDVI_response_depth_QA025_z'] = -(
    analysis['minimum_NDVI_z_QAfrac_ge_025']
    - analysis['prebaseline_NDVI_z']
)

analysis['MOD13_NDVI_response_depth_QA050_z'] = -(
    analysis['minimum_NDVI_z_QAfrac_ge_050']
    - analysis['prebaseline_NDVI_z']
)

# Clean/confounding sensitivities
analysis['GRACE_clean_flag'] = (
    (
        analysis['GRACE_response_metric_available_flag']
        .fillna(0) == 1
    )
    &
    (
        analysis[
            'other_flash_drought_before_minimum_TWSA_flag'
        ].fillna(0) == 0
    )
).astype(int)

analysis['JRC_clean_flag'] = (
    (
        analysis['JRC_response_metric_available_flag']
        .fillna(0) == 1
    )
    &
    (
        analysis[
            'other_flash_drought_before_minimum_JRC_flag'
        ].fillna(0) == 0
    )
).astype(int)

analysis['nonprolonged_flag'] = (
    analysis['prolonged_after_flash_flag'] == 0
).astype(int)

print('Primary response availability:')
for col in [
    'MOD16_ETPET_response_depth_z',
    'MOD13_NDVI_response_depth_z',
    'GRACE_TWSA_response_depth_z',
    'JRC_water_response_depth_z'
]:
    print(col, ':', int(analysis[col].notna().sum()))


## 2. Statistical helper functions


In [ ]:
def bh_adjust(p_values):
    p_values = np.asarray(p_values, dtype=float)
    adjusted = np.full(len(p_values), np.nan, dtype=float)

    valid = np.isfinite(p_values)
    valid_ids = np.where(valid)[0]
    p = p_values[valid]

    if len(p) == 0:
        return adjusted

    order = np.argsort(p)
    ranked = p[order]

    adj_ranked = (
        ranked
        * len(p)
        / np.arange(1, len(p) + 1)
    )

    adj_ranked = np.minimum.accumulate(
        adj_ranked[::-1]
    )[::-1]

    adj_ranked = np.clip(
        adj_ranked,
        0,
        1
    )

    inverse = np.empty(len(p), dtype=int)
    inverse[order] = np.arange(len(p))

    adjusted[valid_ids] = adj_ranked[inverse]

    return adjusted


def spearman_permutation_bootstrap(
    x,
    y,
    n_perm=N_PERM,
    n_boot=N_BOOT,
    seed=42
):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    ok = np.isfinite(x) & np.isfinite(y)

    x = x[ok]
    y = y[ok]

    n = len(x)

    if (
        n < 5
        or np.std(x) == 0
        or np.std(y) == 0
    ):
        return {
            'n': n,
            'rho': np.nan,
            'p_perm': np.nan,
            'ci_low': np.nan,
            'ci_high': np.nan
        }

    rank_x = rankdata(x)
    rank_y = rankdata(y)

    x_centered = rank_x - rank_x.mean()
    y_centered = rank_y - rank_y.mean()

    denominator = np.sqrt(
        np.sum(x_centered ** 2)
        * np.sum(y_centered ** 2)
    )

    rho = float(
        np.sum(x_centered * y_centered)
        / denominator
    )

    rng = np.random.default_rng(seed)

    extreme = 0

    for _ in range(n_perm):
        permuted_y = rng.permutation(
            y_centered
        )

        permuted_rho = float(
            np.sum(
                x_centered
                * permuted_y
            )
            / denominator
        )

        if (
            abs(permuted_rho)
            >= abs(rho) - 1e-15
        ):
            extreme += 1

    p_perm = (
        extreme + 1
    ) / (
        n_perm + 1
    )

    bootstrap_rhos = []

    for _ in range(n_boot):
        idx = rng.integers(
            0,
            n,
            size=n
        )

        xb = x[idx]
        yb = y[idx]

        if (
            np.std(xb) == 0
            or np.std(yb) == 0
        ):
            continue

        r = stats.spearmanr(
            xb,
            yb
        ).statistic

        if np.isfinite(r):
            bootstrap_rhos.append(
                float(r)
            )

    if bootstrap_rhos:
        ci_low, ci_high = np.percentile(
            bootstrap_rhos,
            [2.5, 97.5]
        )
    else:
        ci_low = np.nan
        ci_high = np.nan

    return {
        'n': n,
        'rho': rho,
        'p_perm': float(p_perm),
        'ci_low': float(ci_low),
        'ci_high': float(ci_high)
    }


def spearman_bootstrap_only(
    x,
    y,
    n_boot=N_BOOT,
    seed=42
):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    ok = np.isfinite(x) & np.isfinite(y)

    x = x[ok]
    y = y[ok]

    n = len(x)

    if (
        n < 5
        or np.std(x) == 0
        or np.std(y) == 0
    ):
        return {
            'n': n,
            'rho': np.nan,
            'ci_low': np.nan,
            'ci_high': np.nan
        }

    rho = float(
        stats.spearmanr(
            x,
            y
        ).statistic
    )

    rng = np.random.default_rng(seed)
    bootstrap_rhos = []

    for _ in range(n_boot):
        idx = rng.integers(
            0,
            n,
            size=n
        )

        xb = x[idx]
        yb = y[idx]

        if (
            np.std(xb) == 0
            or np.std(yb) == 0
        ):
            continue

        r = stats.spearmanr(
            xb,
            yb
        ).statistic

        if np.isfinite(r):
            bootstrap_rhos.append(
                float(r)
            )

    if bootstrap_rhos:
        ci_low, ci_high = np.percentile(
            bootstrap_rhos,
            [2.5, 97.5]
        )
    else:
        ci_low = np.nan
        ci_high = np.nan

    return {
        'n': n,
        'rho': rho,
        'ci_low': float(ci_low),
        'ci_high': float(ci_high)
    }


def permutation_kruskal(
    values,
    groups,
    n_perm=N_PERM,
    seed=42
):
    values = np.asarray(
        values,
        dtype=float
    )

    groups = np.asarray(groups)

    ok = (
        np.isfinite(values)
        &
        pd.notna(groups)
    )

    values = values[ok]
    groups = groups[ok]

    unique_groups = np.unique(groups)

    parts = [
        values[groups == g]
        for g in unique_groups
    ]

    sizes = {
        g: int(
            np.sum(groups == g)
        )
        for g in unique_groups
    }

    if (
        len(unique_groups) < 2
        or any(
            len(part) < 2
            for part in parts
        )
    ):
        return {
            'n': len(values),
            'k': len(unique_groups),
            'H': np.nan,
            'p_perm': np.nan,
            'epsilon2': np.nan,
            'group_sizes': str(sizes)
        }

    H = float(
        stats.kruskal(
            *parts
        ).statistic
    )

    n = len(values)
    k = len(unique_groups)

    epsilon2 = (
        max(
            0.0,
            (H - k + 1)
            / (n - k)
        )
        if n > k
        else np.nan
    )

    rng = np.random.default_rng(seed)
    extreme = 0

    for _ in range(n_perm):
        permuted_groups = rng.permutation(
            groups
        )

        permuted_parts = [
            values[
                permuted_groups == g
            ]
            for g in unique_groups
        ]

        permuted_H = float(
            stats.kruskal(
                *permuted_parts
            ).statistic
        )

        if (
            permuted_H
            >= H - 1e-15
        ):
            extreme += 1

    p_perm = (
        extreme + 1
    ) / (
        n_perm + 1
    )

    return {
        'n': n,
        'k': k,
        'H': H,
        'p_perm': float(p_perm),
        'epsilon2': float(epsilon2),
        'group_sizes':
            ';'.join(
                f'{g}:{sizes[g]}'
                for g in unique_groups
            )
    }


## 3. Pre-specified association families


In [ ]:
PRIMARY_PREDICTORS = {
    'P_deficit_strength_z':
        'Precipitation deficit strength',

    'PE_excess_strength_z':
        'Potential evaporation excess',

    'VPD_excess_strength_z':
        'VPD excess',

    'temperature_excess_strength_z':
        'Temperature excess'
}

SECONDARY_PREDICTORS = {
    'radiation_excess_strength_z':
        'Net radiation excess',

    'dry_air_RH_strength_z':
        'Low-RH / dry-air strength',

    'wind_excess_strength_z':
        'Wind excess'
}

EVENT_PHYSICS_PREDICTORS = {
    'severity_40pct_deficit_pp_pentads':
        'Flash-drought severity',

    'mean_intensification_rate_pp_per_pentad':
        'Flash-drought intensification rate'
}

PRIMARY_OUTCOMES = {
    'MOD16_ETPET_response_depth_z':
        'MOD16 ET/PET response depth',

    'MOD13_NDVI_response_depth_z':
        'MOD13 NDVI response depth',

    'GRACE_TWSA_response_depth_z':
        'GRACE TWSA response depth',

    'JRC_water_response_depth_z':
        'JRC surface-water response depth'
}


def association_family(
    df,
    predictors,
    outcomes,
    family_name,
    seed_start
):
    rows = []
    counter = 0

    for predictor, predictor_label in predictors.items():

        for outcome, outcome_label in outcomes.items():

            result = spearman_permutation_bootstrap(
                df[predictor],
                df[outcome],
                seed=seed_start + counter
            )

            rows.append({
                'family':
                    family_name,

                'predictor':
                    predictor,

                'predictor_label':
                    predictor_label,

                'outcome':
                    outcome,

                'outcome_label':
                    outcome_label,

                **result
            })

            counter += 1

    output = pd.DataFrame(rows)

    output['p_fdr_bh'] = bh_adjust(
        output['p_perm']
    )

    output['fdr_0_05_flag'] = (
        output['p_fdr_bh'] <= 0.05
    ).astype(int)

    return output


primary_associations = association_family(
    analysis,
    PRIMARY_PREDICTORS,
    PRIMARY_OUTCOMES,
    'primary_driver_response',
    seed_start=100
)

secondary_associations = association_family(
    analysis,
    SECONDARY_PREDICTORS,
    PRIMARY_OUTCOMES,
    'secondary_atmospheric_response',
    seed_start=1000
)

event_physics_associations = association_family(
    analysis,
    EVENT_PHYSICS_PREDICTORS,
    PRIMARY_OUTCOMES,
    'event_physics_response',
    seed_start=2000
)

display(
    primary_associations.sort_values(
        'p_fdr_bh'
    )
)


## 4. Compound-driver class response comparison


In [ ]:
class_rows = []

for counter, (
    outcome,
    outcome_label
) in enumerate(
    PRIMARY_OUTCOMES.items()
):

    result = permutation_kruskal(
        analysis[outcome],
        analysis['primary_driver_class'],
        seed=3000 + counter
    )

    medians = (
        analysis
        .groupby(
            'primary_driver_class'
        )[outcome]
        .median()
    )

    row = {
        'outcome':
            outcome,

        'outcome_label':
            outcome_label,

        **result
    }

    for group_name, value in medians.items():
        row[
            f'median_{group_name}'
        ] = value

    class_rows.append(row)


driver_class_comparisons = pd.DataFrame(
    class_rows
)

driver_class_comparisons[
    'p_fdr_bh'
] = bh_adjust(
    driver_class_comparisons[
        'p_perm'
    ]
)

driver_class_comparisons[
    'fdr_0_05_flag'
] = (
    driver_class_comparisons[
        'p_fdr_bh'
    ] <= 0.05
).astype(int)

display(driver_class_comparisons)


## 5. Cross-response coupling


In [ ]:
CROSS_RESPONSE_PAIRS = [
    (
        'MOD16_ETPET_response_depth_z',
        'MOD13_NDVI_response_depth_z',
        'MOD16 vs NDVI'
    ),
    (
        'MOD16_ETPET_response_depth_z',
        'GRACE_TWSA_response_depth_z',
        'MOD16 vs TWSA'
    ),
    (
        'MOD16_ETPET_response_depth_z',
        'JRC_water_response_depth_z',
        'MOD16 vs JRC'
    ),
    (
        'MOD13_NDVI_response_depth_z',
        'GRACE_TWSA_response_depth_z',
        'NDVI vs TWSA'
    ),
    (
        'MOD13_NDVI_response_depth_z',
        'JRC_water_response_depth_z',
        'NDVI vs JRC'
    ),
    (
        'GRACE_TWSA_response_depth_z',
        'JRC_water_response_depth_z',
        'TWSA vs JRC'
    )
]

cross_rows = []

for counter, (
    response_x,
    response_y,
    pair_label
) in enumerate(
    CROSS_RESPONSE_PAIRS
):

    result = spearman_permutation_bootstrap(
        analysis[response_x],
        analysis[response_y],
        seed=4000 + counter
    )

    cross_rows.append({
        'pair_label':
            pair_label,

        'response_x':
            response_x,

        'response_y':
            response_y,

        **result
    })


cross_response = pd.DataFrame(
    cross_rows
)

cross_response['p_fdr_bh'] = bh_adjust(
    cross_response['p_perm']
)

cross_response['fdr_0_05_flag'] = (
    cross_response['p_fdr_bh']
    <= 0.05
).astype(int)

display(
    cross_response.sort_values(
        'p_fdr_bh'
    )
)


## 6. Robustness sensitivities

These are effect-size/CI diagnostics.

They are **not** treated as a new confirmatory hypothesis family.


In [ ]:
robustness_rows = []
counter = 0

driver_sensitivity_specs = [
    (
        'MOD16 primary all',
        'MOD16_ETPET_response_depth_z',
        pd.Series(True, index=analysis.index)
    ),
    (
        'MOD16 strictQA all',
        'MOD16_ETPET_response_depth_strictQA_z',
        pd.Series(True, index=analysis.index)
    ),
    (
        'MOD16 primary nonprolonged',
        'MOD16_ETPET_response_depth_z',
        analysis['nonprolonged_flag'] == 1
    ),
    (
        'MOD13 primary all',
        'MOD13_NDVI_response_depth_z',
        pd.Series(True, index=analysis.index)
    ),
    (
        'MOD13 QA>=0.25 minimum',
        'MOD13_NDVI_response_depth_QA025_z',
        pd.Series(True, index=analysis.index)
    ),
    (
        'MOD13 QA>=0.50 minimum',
        'MOD13_NDVI_response_depth_QA050_z',
        pd.Series(True, index=analysis.index)
    ),
    (
        'MOD13 primary nonprolonged',
        'MOD13_NDVI_response_depth_z',
        analysis['nonprolonged_flag'] == 1
    ),
    (
        'GRACE all baseline-relative',
        'GRACE_TWSA_response_depth_z',
        pd.Series(True, index=analysis.index)
    ),
    (
        'GRACE clean-confounding',
        'GRACE_TWSA_response_depth_z',
        analysis['GRACE_clean_flag'] == 1
    ),
    (
        'GRACE nonprolonged',
        'GRACE_TWSA_response_depth_z',
        analysis['nonprolonged_flag'] == 1
    ),
    (
        'JRC all baseline-relative',
        'JRC_water_response_depth_z',
        pd.Series(True, index=analysis.index)
    ),
    (
        'JRC clean-confounding',
        'JRC_water_response_depth_z',
        analysis['JRC_clean_flag'] == 1
    ),
    (
        'JRC nonprolonged',
        'JRC_water_response_depth_z',
        analysis['nonprolonged_flag'] == 1
    )
]

for (
    variant,
    outcome,
    mask
) in driver_sensitivity_specs:

    subset = analysis.loc[
        mask
    ]

    for (
        predictor,
        predictor_label
    ) in [
        (
            'P_deficit_strength_z',
            'Precipitation deficit strength'
        ),
        (
            'PE_excess_strength_z',
            'Potential evaporation excess'
        )
    ]:

        result = spearman_bootstrap_only(
            subset[predictor],
            subset[outcome],
            seed=5000 + counter
        )

        robustness_rows.append({
            'analysis_type':
                'driver_response_sensitivity',

            'variant':
                variant,

            'predictor':
                predictor,

            'predictor_label':
                predictor_label,

            'outcome':
                outcome,

            **result,

            'median_response_depth_z':
                float(
                    pd.to_numeric(
                        subset[outcome],
                        errors='coerce'
                    ).median()
                )
        })

        counter += 1


cross_sensitivity_specs = [
    (
        'MOD16-NDVI primary',
        'MOD16_ETPET_response_depth_z',
        'MOD13_NDVI_response_depth_z',
        pd.Series(True, index=analysis.index)
    ),
    (
        'MOD16-NDVI NDVI_QA025',
        'MOD16_ETPET_response_depth_z',
        'MOD13_NDVI_response_depth_QA025_z',
        pd.Series(True, index=analysis.index)
    ),
    (
        'MOD16-NDVI NDVI_QA050',
        'MOD16_ETPET_response_depth_z',
        'MOD13_NDVI_response_depth_QA050_z',
        pd.Series(True, index=analysis.index)
    ),
    (
        'MOD16-NDVI nonprolonged',
        'MOD16_ETPET_response_depth_z',
        'MOD13_NDVI_response_depth_z',
        analysis['nonprolonged_flag'] == 1
    ),
    (
        'TWSA-JRC all',
        'GRACE_TWSA_response_depth_z',
        'JRC_water_response_depth_z',
        pd.Series(True, index=analysis.index)
    ),
    (
        'TWSA-JRC both clean',
        'GRACE_TWSA_response_depth_z',
        'JRC_water_response_depth_z',
        (
            (analysis['GRACE_clean_flag'] == 1)
            &
            (analysis['JRC_clean_flag'] == 1)
        )
    ),
    (
        'TWSA-JRC nonprolonged',
        'GRACE_TWSA_response_depth_z',
        'JRC_water_response_depth_z',
        analysis['nonprolonged_flag'] == 1
    )
]

for (
    variant,
    response_x,
    response_y,
    mask
) in cross_sensitivity_specs:

    subset = analysis.loc[
        mask
    ]

    result = spearman_bootstrap_only(
        subset[response_x],
        subset[response_y],
        seed=6000 + counter
    )

    robustness_rows.append({
        'analysis_type':
            'cross_response_sensitivity',

        'variant':
            variant,

        'predictor':
            response_x,

        'predictor_label':
            'response_x',

        'outcome':
            response_y,

        **result,

        'median_response_depth_z':
            np.nan
    })

    counter += 1


robustness = pd.DataFrame(
    robustness_rows
)

display(robustness)


## 7. Response cohort summary


In [ ]:
cohort_rows = []


def add_cohort(
    cohort_name,
    mask,
    outcome
):
    values = analysis.loc[
        mask,
        outcome
    ]

    cohort_rows.append({
        'cohort':
            cohort_name,

        'outcome':
            outcome,

        'n_events':
            int(
                values.notna().sum()
            ),

        'median_response_depth_z':
            (
                float(
                    values.median()
                )
                if values.notna().any()
                else np.nan
            )
    })


all_events = pd.Series(
    True,
    index=analysis.index
)

add_cohort(
    'MOD16 primary',
    all_events,
    'MOD16_ETPET_response_depth_z'
)

add_cohort(
    'MOD16 strictQA',
    all_events,
    'MOD16_ETPET_response_depth_strictQA_z'
)

add_cohort(
    'MOD13 primary',
    all_events,
    'MOD13_NDVI_response_depth_z'
)

add_cohort(
    'MOD13 QA025 sensitivity',
    all_events,
    'MOD13_NDVI_response_depth_QA025_z'
)

add_cohort(
    'MOD13 QA050 sensitivity',
    all_events,
    'MOD13_NDVI_response_depth_QA050_z'
)

add_cohort(
    'GRACE baseline-relative',
    all_events,
    'GRACE_TWSA_response_depth_z'
)

add_cohort(
    'GRACE clean-confounding',
    analysis['GRACE_clean_flag'] == 1,
    'GRACE_TWSA_response_depth_z'
)

add_cohort(
    'JRC baseline-relative',
    all_events,
    'JRC_water_response_depth_z'
)

add_cohort(
    'JRC clean-confounding',
    analysis['JRC_clean_flag'] == 1,
    'JRC_water_response_depth_z'
)

cohort_summary = pd.DataFrame(
    cohort_rows
)

display(cohort_summary)


## 8. Phase 2G QC


In [ ]:
qc_rows = []


def add_qc(
    check,
    value,
    status,
    note
):
    qc_rows.append({
        'check':
            check,

        'value':
            value,

        'status':
            status,

        'note':
            note
    })


add_qc(
    'Events retained',
    len(analysis),
    'PASS',
    'Must remain 39.'
)

add_qc(
    'Unique event IDs',
    analysis['event_id'].nunique(),
    'PASS',
    'Must remain 39.'
)

add_qc(
    'MOD16 primary response depth n',
    int(
        analysis[
            'MOD16_ETPET_response_depth_z'
        ].notna().sum()
    ),
    'PASS',
    'Expected 39.'
)

add_qc(
    'MOD13 primary response depth n',
    int(
        analysis[
            'MOD13_NDVI_response_depth_z'
        ].notna().sum()
    ),
    'PASS',
    'Expected 39.'
)

add_qc(
    'GRACE baseline-relative response depth n',
    int(
        analysis[
            'GRACE_TWSA_response_depth_z'
        ].notna().sum()
    ),
    'RESULT',
    'Missing source/baseline support remains missing.'
)

add_qc(
    'JRC baseline-relative response depth n',
    int(
        analysis[
            'JRC_water_response_depth_z'
        ].notna().sum()
    ),
    'RESULT',
    'Missing/coverage-limited support remains missing.'
)

add_qc(
    'Primary association tests',
    len(primary_associations),
    'PASS',
    'Pre-specified family must contain 16 tests.'
)

add_qc(
    'Primary FDR q<=0.05 results',
    int(
        primary_associations[
            'fdr_0_05_flag'
        ].sum()
    ),
    'RESULT',
    'Do not tune tests based on this result.'
)

add_qc(
    'Secondary FDR q<=0.05 results',
    int(
        secondary_associations[
            'fdr_0_05_flag'
        ].sum()
    ),
    'RESULT',
    'Exploratory family.'
)

add_qc(
    'Event-physics FDR q<=0.05 results',
    int(
        event_physics_associations[
            'fdr_0_05_flag'
        ].sum()
    ),
    'RESULT',
    'Separate physical family.'
)

add_qc(
    'Driver-class omnibus FDR q<=0.05 results',
    int(
        driver_class_comparisons[
            'fdr_0_05_flag'
        ].sum()
    ),
    'RESULT',
    'No automatic pairwise post-hoc tests.'
)

add_qc(
    'Cross-response FDR q<=0.05 results',
    int(
        cross_response[
            'fdr_0_05_flag'
        ].sum()
    ),
    'RESULT',
    'Interpret effect size and CI.'
)

add_qc(
    'Prolonged events',
    int(
        analysis[
            'prolonged_after_flash_flag'
        ].sum()
    ),
    'RESULT',
    'Retained in primary; excluded only in sensitivity.'
)

add_qc(
    'GRACE clean-confounding baseline-relative n',
    int(
        analysis.loc[
            analysis['GRACE_clean_flag'] == 1,
            'GRACE_TWSA_response_depth_z'
        ].notna().sum()
    ),
    'RESULT',
    'Sensitivity cohort.'
)

add_qc(
    'JRC clean-confounding baseline-relative n',
    int(
        analysis.loc[
            analysis['JRC_clean_flag'] == 1,
            'JRC_water_response_depth_z'
        ].notna().sum()
    ),
    'RESULT',
    'Sensitivity cohort.'
)

qc = pd.DataFrame(qc_rows)

display(qc)

assert len(analysis) == 39
assert analysis['event_id'].nunique() == 39
assert len(primary_associations) == 16
assert not analysis.columns.duplicated().any()


## 9. Save Phase 2G tables automatically


In [ ]:
ANALYSIS_OUT = PHASE2G_DIR / 'AEZ26_Phase2G_Event_Statistical_Analysis_Table.csv'
PRIMARY_OUT = PHASE2G_DIR / 'AEZ26_Phase2G_Primary_Driver_Response_Associations.csv'
SECONDARY_OUT = PHASE2G_DIR / 'AEZ26_Phase2G_Secondary_Atmospheric_Response_Associations.csv'
EVENTPHYS_OUT = PHASE2G_DIR / 'AEZ26_Phase2G_EventPhysics_Response_Associations.csv'
CLASS_OUT = PHASE2G_DIR / 'AEZ26_Phase2G_DriverClass_Response_Comparisons.csv'
CROSS_OUT = PHASE2G_DIR / 'AEZ26_Phase2G_Cross_Response_Coupling.csv'
ROBUST_OUT = PHASE2G_DIR / 'AEZ26_Phase2G_Robustness_Sensitivity.csv'
COHORT_OUT = PHASE2G_DIR / 'AEZ26_Phase2G_Response_Cohort_Summary.csv'
QC_OUT = PHASE2G_DIR / 'AEZ26_Phase2G_QC.csv'
CONFIG_OUT = PHASE2G_DIR / 'AEZ26_Phase2G_Method_Config.txt'

analysis.to_csv(
    ANALYSIS_OUT,
    index=False,
    date_format='%Y-%m-%d'
)

primary_associations.to_csv(
    PRIMARY_OUT,
    index=False
)

secondary_associations.to_csv(
    SECONDARY_OUT,
    index=False
)

event_physics_associations.to_csv(
    EVENTPHYS_OUT,
    index=False
)

driver_class_comparisons.to_csv(
    CLASS_OUT,
    index=False
)

cross_response.to_csv(
    CROSS_OUT,
    index=False
)

robustness.to_csv(
    ROBUST_OUT,
    index=False
)

cohort_summary.to_csv(
    COHORT_OUT,
    index=False
)

qc.to_csv(
    QC_OUT,
    index=False
)

method_config = f'''PHASE 2G METHOD CONFIG
Primary events: 39
Flash-drought labels changed: NO

Primary response depth sign convention:
positive = stronger decline relative to Phase 2F pre-event baseline
negative values are retained, not clipped

Primary predictor family:
- precipitation deficit strength = -P_z
- PE excess = PE_z
- VPD excess = VPD_z
- temperature excess = T_z

Primary response family:
- MOD16 ET/PET depth
- MOD13 NDVI depth
- GRACE TWSA depth
- JRC surface-water depth

Association statistic:
- Spearman rho
- two-sided permutation p
- permutations = {N_PERM}
- percentile bootstrap 95% CI
- bootstrap resamples = {N_BOOT}

Multiplicity:
- Benjamini-Hochberg FDR
- applied separately within pre-specified families

Driver-class comparison:
- Kruskal-Wallis H
- permutation p
- epsilon-squared effect size
- no automatic pairwise post-hoc tests

Robustness:
- MOD16 strict QA
- MOD13 QA >=0.25 / >=0.50
- no-prolonged sensitivity
- clean-confounding GRACE/JRC sensitivity

Groundwater:
- NOT analyzed in Phase 2G
- GRACE remains TWSA, not direct groundwater

No final ML/PIML fitting in Phase 2G.
'''

CONFIG_OUT.write_text(
    method_config,
    encoding='utf-8'
)

print('Saved Phase 2G tables.')


## 10. Diagnostic figures


In [ ]:
# Figure 1 — primary driver-response Spearman rho heatmap
rho_table = (
    primary_associations
    .pivot(
        index='predictor_label',
        columns='outcome_label',
        values='rho'
    )
)

plt.figure(figsize=(10, 6))

image = plt.imshow(
    rho_table.to_numpy(),
    aspect='auto',
    vmin=-1,
    vmax=1
)

plt.colorbar(
    image,
    label='Spearman rho'
)

plt.xticks(
    range(len(rho_table.columns)),
    rho_table.columns,
    rotation=30,
    ha='right'
)

plt.yticks(
    range(len(rho_table.index)),
    rho_table.index
)

plt.title(
    'AEZ-26 Primary Driver–Response Associations'
)

plt.tight_layout()

plt.savefig(
    FIG_DIR / 'AEZ26_Phase2G_Primary_Driver_Response_Rho_Heatmap.png',
    dpi=200,
    bbox_inches='tight'
)

plt.show()


In [ ]:
# Figure 2 — cross-response Spearman rho
plt.figure(figsize=(9, 5))

x_positions = np.arange(
    len(cross_response)
)

plt.bar(
    x_positions,
    cross_response['rho']
)

plt.axhline(0)

plt.xticks(
    x_positions,
    cross_response['pair_label'],
    rotation=30,
    ha='right'
)

plt.ylabel('Spearman rho')
plt.title(
    'AEZ-26 Cross-Response Event-Scale Coupling'
)

plt.tight_layout()

plt.savefig(
    FIG_DIR / 'AEZ26_Phase2G_Cross_Response_Rho.png',
    dpi=200,
    bbox_inches='tight'
)

plt.show()


In [ ]:
# Figure 3 — MOD16 versus NDVI response depth
plot_df = analysis.dropna(
    subset=[
        'MOD16_ETPET_response_depth_z',
        'MOD13_NDVI_response_depth_z'
    ]
)

plt.figure(figsize=(7, 6))

plt.scatter(
    plot_df[
        'MOD16_ETPET_response_depth_z'
    ],
    plot_df[
        'MOD13_NDVI_response_depth_z'
    ]
)

plt.axhline(0)
plt.axvline(0)

plt.xlabel(
    'MOD16 ET/PET response depth (z)'
)

plt.ylabel(
    'MOD13 NDVI response depth (z)'
)

plt.title(
    'AEZ-26 Functional vs Vegetation Response Depth'
)

plt.tight_layout()

plt.savefig(
    FIG_DIR / 'AEZ26_Phase2G_MOD16_vs_NDVI_Response_Depth.png',
    dpi=200,
    bbox_inches='tight'
)

plt.show()


## 11. Create one ZIP audit package


In [ ]:
ZIP_BASE = PHASE2_DIR / 'AEZ26_Phase2G_Output_Package'

zip_path = shutil.make_archive(
    str(ZIP_BASE),
    'zip',
    root_dir=PHASE2_DIR,
    base_dir='Phase2G'
)

print('ZIP CREATED:')
print(zip_path)


In [ ]:
print('FINAL PHASE 2G CHECK')
print('Events:', len(analysis))
print(
    'Primary FDR q<=0.05:',
    int(
        primary_associations[
            'fdr_0_05_flag'
        ].sum()
    )
)
print(
    'Driver-class FDR q<=0.05:',
    int(
        driver_class_comparisons[
            'fdr_0_05_flag'
        ].sum()
    )
)
print(
    'Cross-response FDR q<=0.05:',
    int(
        cross_response[
            'fdr_0_05_flag'
        ].sum()
    )
)

print()
print('Top primary associations:')
display(
    primary_associations.sort_values(
        'p_fdr_bh'
    ).head(8)
)

print()
print('Cross-response coupling:')
display(
    cross_response.sort_values(
        'p_fdr_bh'
    )
)

print()
print('Upload this ZIP for independent audit:')
print(zip_path)


## Next gate

Upload the Phase 2G ZIP for independent audit.

After Phase 2G passes, the workflow splits into two parallel branches:

### Groundwater branch
Acquire and QC BWDB/BMDA observation-well data, then create groundwater event-response metrics.

### Modeling branch
Construct leakage-safe temporal model-ready datasets and baseline ML experiments using only targets that are scientifically supported.

Direct groundwater-target modeling should wait for defensible well observations.
